In [1]:
# Project Name: Employee Attrition Analyzer 
# Name: SYED MUHAMMAD AMMAR SHAH   Roll No: 24F-Ai-66
# Course: Machine Learning  Lab 7: Open Ended Lab
# Objective: Predict whether an employee will leave the company (Attrition = Yes/No) using Logistic Regression, Decision Tree and Random Forest.

In [ ]:
#  Import libraries 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

In [ ]:
#  Load dataset
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
#basic checks
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nAttrition counts:\n", df["Attrition"].value_counts())


In [ ]:
# Class distribution
sns.countplot(x="Attrition", data=df)
plt.title("Attrition Distribution (Imbalanced Data)")
plt.show()

In [ ]:
# Preprocessing steps:
#1. Drop useless columns
df = df.drop(columns=["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"])

#2. Encode target column
y = df["Attrition"].map({"Yes": 1, "No": 0})

#3. One-Hot Encoding for categorical columns
X = pd.get_dummies(df.drop("Attrition", axis=1), drop_first=True, dtype=int)
print("Features after encoding:", X.shape[1])

#4. Train-Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
#Model Training
# class_weight="balanced" is used because only ~16% employees left,
# so models should not ignore the minority class.

#Model 1: Logistic Regression
log_model = LogisticRegression(max_iter=1000, class_weight="balanced")
log_model.fit(X_train_scaled, y_train)
log_pred = log_model.predict(X_test_scaled)

#Model 2: Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

#Model 3: Random Forest
rf_model = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=10,
                                  class_weight="balanced_subsample", random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("All 3 models trained.")

In [ ]:
#Metrics for each model
predictions = {
    "Logistic Regression": log_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
}

rows = []
for name, pred in predictions.items():
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

In [ ]:
# Classification report of each model
for name, pred in predictions.items():
    print("=" * 55)
    print(name)
    print(classification_report(y_test, pred, target_names=["Stayed", "Left"]))

In [ ]:
#Confusion matrices 
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                           display_labels=["Stayed", "Left"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
#Model comparison chart
results.plot(kind="bar", figsize=(9, 5))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.show()

In [ ]:
#important features (Random Forest)
rf_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
sns.barplot(x=rf_imp.values, y=rf_imp.index)
plt.title("Top 10 Features - Random Forest")
plt.xlabel("Importance")
plt.show()

#important features (Decision Tree)
dt_imp = pd.Series(dt_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
sns.barplot(x=dt_imp.values, y=dt_imp.index)
plt.title("Top 10 Features - Decision Tree")
plt.xlabel("Importance")
plt.show()